In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: una Qiskit Function di Qedma
*Consulta il [riferimento API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti dei piani IBM Quantum&reg; Premium, Flex e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Sebbene le unità di elaborazione quantistica abbiano fatto grandi progressi negli ultimi anni, gli errori dovuti al rumore e alle imperfezioni dell'hardware esistente rimangono una sfida centrale per gli sviluppatori di algoritmi quantistici. Con l'avvicinarsi di computazioni quantistiche su scala utilitaria che non possono essere verificate classicamente, le soluzioni per eliminare il rumore con precisione garantita diventano sempre più importanti. Per superare questa sfida, Qedma ha sviluppato Quantum Error Mitigation (QESEM), integrato senza soluzione di continuità su IBM Quantum Platform come [Qiskit Function](/guides/functions).

Con QESEM, gli utenti possono eseguire i propri circuiti quantistici su QPU rumorose per ottenere risultati altamente accurati e privi di errori, con overhead di tempo QPU estremamente efficienti, vicini ai limiti fondamentali. Per raggiungere questo obiettivo, QESEM sfrutta una serie di metodi proprietari sviluppati da Qedma per la caratterizzazione e la riduzione degli errori. Le tecniche di riduzione degli errori includono l'ottimizzazione dei gate, la transpilazione consapevole del rumore, la soppressione degli errori (ES) e la mitigazione degli errori non distorta (EM). Grazie a questa combinazione di metodi basati sulla caratterizzazione, gli utenti possono ottenere risultati affidabili e privi di errori per circuiti quantistici generici ad alto volume, aprendo la strada ad applicazioni altrimenti impossibili da realizzare.

Per una descrizione completa dei componenti sottostanti e una dimostrazione su scala utilitaria, consulta l'articolo [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Descrizione
Puoi utilizzare la funzione QESEM di Qedma per stimare ed eseguire facilmente i tuoi circuiti con soppressione e mitigazione degli errori, ottenendo volumi di circuiti più grandi e precisioni più elevate. Per usare QESEM, devi fornire un circuito quantistico, un insieme di osservabili da misurare, una precisione statistica target per ciascun osservabile e una QPU scelta. Prima di eseguire il circuito alla precisione target, puoi stimare il tempo QPU necessario sulla base di un calcolo analitico che non richiede l'esecuzione del circuito. Una volta soddisfatto della stima del tempo QPU, puoi eseguire il circuito con QESEM.

Quando si esegue un circuito, QESEM avvia un protocollo di caratterizzazione del dispositivo personalizzato per il circuito, producendo un modello di rumore affidabile per gli errori che si verificano nel circuito. Sulla base della caratterizzazione, QESEM implementa innanzitutto la transpilazione consapevole del rumore per mappare il circuito di input su un insieme di qubit fisici e gate, minimizzando il rumore che influisce sull'osservabile target. Questi includono i gate nativamente disponibili (CX/CZ sui dispositivi IBM&reg;), oltre a gate aggiuntivi ottimizzati da QESEM, che formano il set di gate esteso di QESEM. QESEM esegue quindi un insieme di circuiti ES ed EM basati sulla caratterizzazione sulla QPU e raccoglie i risultati delle misurazioni. Questi vengono poi post-elaborati classicamente per fornire un valore di aspettativa non distorto e una barra di errore per ciascun osservabile, corrispondente alla precisione richiesta.

![Panoramica di Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM ha dimostrato di fornire risultati ad alta precisione per un'ampia varietà di applicazioni quantistiche e sui volumi di circuiti più grandi attualmente raggiungibili. QESEM offre le seguenti funzionalità orientate all'utente, dimostrate nella sezione benchmark qui sotto:
-	**Precisione garantita:** QESEM produce stime non distorte per i valori di aspettativa degli osservabili. Il suo metodo EM è dotato di garanzie teoriche che, insieme alla caratterizzazione all'avanguardia di Qedma, assicurano che la mitigazione converga all'output del circuito privo di rumore entro la precisione specificata dall'utente. A differenza di molti metodi EM euristici soggetti a errori sistematici o distorsioni, la precisione garantita di QESEM è essenziale per garantire risultati affidabili in circuiti quantistici e osservabili generici.
-	**Scalabilità a QPU di grandi dimensioni:** il tempo QPU di QESEM dipende dai volumi dei circuiti, ma è altrimenti indipendente dal numero di qubit. Qedma ha dimostrato QESEM sui dispositivi quantistici più grandi disponibili oggi, inclusi IBM Quantum Eagle a 127 qubit e Heron a 133 qubit.
-	**Indipendente dall'applicazione:** QESEM è stato dimostrato su un'ampia varietà di applicazioni, tra cui simulazione hamiltoniana, VQE, QAOA e stima dell'ampiezza. Gli utenti possono inserire qualsiasi circuito quantistico e osservabile da misurare e ottenere risultati accurati e privi di errori. Le uniche limitazioni sono dettate dalle specifiche hardware e dal tempo QPU allocato, che determinano i volumi di circuiti accessibili e le precisioni di output. Al contrario, molte soluzioni di riduzione degli errori sono specifiche per applicazione o coinvolgono euristiche non controllate, rendendole inapplicabili per circuiti e applicazioni quantistiche generiche.
-  **Set di gate esteso:** QESEM supporta i gate ad angolo frazionario e fornisce gate $Rzz(\theta)$ ad angolo frazionario ottimizzati da Qedma sui dispositivi IBM Quantum Heron e Eagle. Questo set di gate esteso consente una compilazione più efficiente e sblocca volumi di circuiti superiori fino a un fattore 2 rispetto alla compilazione CX/CZ predefinita.
-	**Osservabili multi-base:** QESEM supporta osservabili di input composti da molte stringhe di Pauli non commutanti, come gli Hamiltoniani generici. La scelta delle basi di misura e l'ottimizzazione dell'allocazione delle risorse QPU (shot e circuiti) vengono quindi eseguite automaticamente da QESEM per minimizzare il tempo QPU richiesto per la precisione richiesta. Questa ottimizzazione, che tiene conto delle fedeltà hardware e dei tassi di esecuzione, consente di eseguire circuiti più profondi e ottenere precisioni più elevate.
## Benchmark
QESEM è stato testato su un'ampia varietà di casi d'uso e applicazioni. I seguenti esempi possono aiutarti a valutare quali tipi di carichi di lavoro puoi eseguire con QESEM.

Una figura di merito fondamentale per quantificare la difficoltà sia della mitigazione degli errori sia della simulazione classica per un dato circuito e osservabile è il **volume attivo**: il numero di gate CNOT che influiscono sull'osservabile nel circuito. Il volume attivo dipende dalla profondità e dalla larghezza del circuito, dal peso dell'osservabile e dalla struttura del circuito, che determina il cono di luce dell'osservabile. Per ulteriori dettagli, consulta il talk dell'[IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM offre un valore particolarmente elevato nel regime ad alto volume, fornendo risultati affidabili per circuiti e osservabili generici.

![Volume attivo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Applicazione    | Numero di qubit | Dispositivo | Descrizione del circuito | Precisione | Tempo totale | Utilizzo runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 strati totali, 9 basi di misura, catena 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 strati unici x 3 passi, topologia heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 strati unici x 8 passi, topologia heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulazione hamiltoniana di Trotter   | 40  | Eagle (r3)            | 2 strati unici x 10 passi di Trotter, catena 1D                    | 97%      | 3 ore     | 25 min         |
| Simulazione hamiltoniana di Trotter   | 119  | Eagle (r3)           | 3 strati unici x 9 passi di Trotter, topologia heavy-hex 2D                    | 95%      | 6,5 ore     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 strati unici x 15 passi, topologia heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

La precisione è misurata qui rispetto al valore ideale dell'osservabile: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, dove '$\epsilon$' è la precisione assoluta della mitigazione (impostata dall'input dell'utente) e $\langle O \rangle_{ideal}$ è l'osservabile nel circuito privo di rumore.
"Utilizzo runtime" misura l'utilizzo del benchmark in modalità batch (somma dell'utilizzo dei singoli job), mentre "tempo totale" misura l'utilizzo in modalità sessione (tempo reale dell'esperimento), che include i tempi di elaborazione classica e comunicazione aggiuntivi. QESEM è disponibile per l'esecuzione in entrambe le modalità, in modo che gli utenti possano fare il miglior uso delle risorse disponibili.

I circuiti Kicked Ising a 28 qubit simulano il Quasicristallo a Tempo Discreto studiato da Shinjo et al. (vedi [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) e [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) su tre loop connessi di ibm_kawasaki. I parametri del circuito utilizzati qui sono $(\theta_x, \theta_z) = (0.9 \pi, 0)$, con uno stato iniziale ferromagnetico $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. L'osservabile misurato è il valore assoluto della magnetizzazione $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. L'esperimento Kicked Ising su scala utilitaria è stato eseguito sui 136 migliori qubit di ibm_fez; questo benchmark in particolare è stato eseguito all'angolo di Clifford $(\theta_x, \theta_z) = (\pi, 0)$, al quale il volume attivo cresce lentamente con la profondità del circuito, il che — insieme alle elevate fedeltà del dispositivo — consente alta precisione con un breve tempo di esecuzione.

I circuiti di simulazione hamiltoniana di Trotter sono relativi a un modello di Ising a Campo Trasverso ad angoli frazionari: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ e $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ rispettivamente (vedi [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Il circuito su scala utilitaria è stato eseguito sui 119 migliori qubit di ibm_brisbane, mentre l'esperimento a 40 qubit è stato eseguito sulla migliore catena disponibile. La precisione è riportata per la magnetizzazione; risultati ad alta precisione sono stati ottenuti anche per osservabili di peso maggiore.

Il circuito VQE è stato sviluppato insieme a ricercatori del Center for Quantum Technology and Applications presso il Deutsches Elektronen-Synchrotron (DESY). L'osservabile target era un Hamiltoniano composto da un grande numero di stringhe di Pauli non commutanti, sottolineando le prestazioni ottimizzate di QESEM per gli osservabili multi-base. La mitigazione è stata applicata a un ansatz ottimizzato classicamente; sebbene questi risultati siano ancora inediti, risultati della stessa qualità si otterranno per circuiti diversi con proprietà strutturali simili.
## Per iniziare
Autenticati con la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function QESEM come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nel tuo ambiente locale.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Puoi usare le familiari API di Qiskit Serverless per controllare lo stato del tuo carico di lavoro Qiskit Function o recuperare i risultati:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Il seguente snippet di codice mostra come recuperare la stima del tempo QPU (quando `estimate_time_only` è impostato):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Il seguente snippet di codice mostra come recuperare i risultati della mitigazione (quando `estimate_time_only` non è impostato) e le metriche di esecuzione. Questi contengono dati essenziali che consentono una comprensione più approfondita di come i diversi parametri influiscono sull'esecuzione di QESEM. Può essere utile anche quando scrivi un articolo basato sulla tua ricerca.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Recupero dei messaggi di errore
Se lo stato del tuo carico di lavoro è ERROR, usa `job.result()` per recuperare il messaggio di errore come segue:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Supporto

Il team di supporto di Qedma è qui per aiutarti! Se riscontri problemi o hai domande sull'uso della Qiskit Function QESEM, non esitare a contattarci. Il nostro personale di supporto qualificato e disponibile è pronto ad assisterti per qualsiasi questione tecnica o richiesta.

Puoi inviarci un'e-mail a support@qedma.com per ricevere assistenza. Includi quanti più dettagli possibile sul problema che stai riscontrando per aiutarci a fornire una risposta rapida e precisa. Puoi anche contattare il tuo referente dedicato Qedma (POC) tramite e-mail o telefono.

Per aiutarci ad assisterti in modo più efficiente, fornisci le seguenti informazioni quando ci contatti:

- Una descrizione dettagliata del problema
- L'ID del job
- Eventuali messaggi o codici di errore pertinenti

Ci impegniamo a fornirti un supporto rapido ed efficace per garantirti la migliore esperienza possibile con la nostra Qiskit Function.

Siamo sempre alla ricerca di miglioramenti per il nostro prodotto e accogliamo con piacere i tuoi suggerimenti! Se hai idee su come possiamo migliorare i nostri servizi o funzionalità che vorresti vedere, inviaci i tuoi pensieri a support@qedma.com.

## Passi successivi

> **Tip:** - [Richiedi l'accesso a Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Consulta il [riferimento API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) per questa Qiskit Function.
> - Leggi [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Leggi [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).